In [1]:
%pip install pgmpy

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import warnings
warnings.filterwarnings("ignore")   # hide library notices

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

# 2) P(Rain): no parents, so one column
rain = TabularCPD(
    variable="rain", variable_card=3,
    values=[[0.7],    # none
            [0.2],    # light
            [0.1]],   # heavy
    state_names={"rain": ["none", "light", "heavy"]})

# 3) P(Maintenance | Rain): one column for each value of rain
#                rain = none  light  heavy
maintenance = TabularCPD(
    variable="maintenance", variable_card=2,
    values=[[0.4, 0.2, 0.1],    # yes
            [0.6, 0.8, 0.9]],   # no
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]})

# 4) P(Train | Rain, Maintenance): one column for each (rain, maintenance) pair
#     (none,yes) (none,no) (light,yes) (light,no) (heavy,yes) (heavy,no)
train = TabularCPD(
    variable="train", variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],    # on time
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],   # delayed
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]})

# 5) P(Appointment | Train): one column for each value of train
#                train = on time  delayed
appointment = TabularCPD(
    variable="appointment", variable_card=2,
    values=[[0.9, 0.6],    # attend
            [0.1, 0.4]],   # miss
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]})

# 6) Put the tables into the graph and check them
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


In [2]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


In [3]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="heavy")
     * maintenance.get_value(maintenance="yes", rain="heavy")
     * train.get_value(train="delayed", rain="heavy", maintenance="yes")
     * appointment.get_value(appointment="miss", train="delayed"))
print(round(p, 4))

0.0024


In [4]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [5]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


In [6]:
show({"train": "delayed", "rain": "heavy"})

rain: heavy   (observed)
maintenance
    yes: 0.1176
    no: 0.8824
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


Hand-solved Problems (3.6, 3.7, 4.8, 6.10, 6.11, 6.12)
--------------------------------------------------------
P(r, m, t, a) = P(r) × P(m | r) × P(t | r, m) × P(a | t)
P(X | e) = α P(X, e) = α Σy P(X, e, y)
--------------------------------------------------------

Part 3 Question 6-7
3.6
- Query: P(rain = none, maintenance = no, train = on time, appointment = attend) = 0.7 x 0.6 x 0.9 x 0.9
3.7
- Solution: 0.7 x 0.6 x 0.9 x 0.9 = 0.3402

Part 4 Question 8
4.8
- Query: P(rain = heavy, maintenance = yes, train = delayed, appointment = miss)
- Solution: 0.1 x 0.1 x 0.6 x 0.4 = 0.0024

Part 6 Question 10-12
6.10
- P(none, delayed) = 0.7 [P(none)] x (0.4 [P(none,yes,delayed)] x 0.2 [P(none,yes)] + 0.6 x 0.1) = 0.7 x (0.08 + 0.06) = 0.7 x 0.14 = 0.098
- P(light, delayed) = 0.2 x (0.2 x 0.4 + 0.8 x 0.3) = 0.2 x (0.08 + 0.24) = 0.2 x 0.32 = 0.064
- P(heavy, delayed) = 0.1 x (0.1 x 0.6 + 0.9 x 0.5) = 0.1 x (0.06 + 0.45) = 0.1 x 0.51 = 0.051
6.11
- Sum of the unnormalized values = 0.098 + 0.064 + 0.051 = 0.213
6.12
Normalize P(rain = none | train = delayed) => answer/sum
- 0.098/0.213 = 0.4601

--------------------------------------------------------
Short-Answer Problems (13, 14, 16, 18)
--------------------------------------------------------

Question 13:
- Both the code and hand calculation use the same probability rules, only the code reuses already found/partial results, yielding a faster output.

Question 14: 
- Appointment only depends on its direct parent 'Train'. Once the value of train = delayed is obtained, the values of rain and maintenance are no longer needed, hence why the result (0.4) is read directly from the appointments table.

Question 16:
- Missing an appointment makes a delay more likely, which in turn means rain is more probable.

Question 18:
- Without maintenance the train was already likely to be delayed by 50% under heavy rain, and 60% with, therefore the main contributing factor to its delay is the heavy rain. Hence, finding out that the train was delayed doesn't alter the likelihood of a maintenance occurrence much, since it doesn't contribute much to the delay in the presence of heavy rain.